# Standard Gaussian Naive Bayes - Gallstone Dataset\n\nTrains a standard (non-private) Gaussian Naive Bayes classifier on the Gallstone dataset.


## 1. Import Libraries


In [1]:
import time
import numpy as np
import pandas as pd
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import pickle
import json
import os
import warnings
warnings.filterwarnings('ignore')

# Shared metric collector (repository root): logs training accuracy,
# balanced accuracy and AUROC alongside the metrics already tracked below.
import os as _os
import sys as _sys
_REPO_ROOT = _os.path.abspath(_os.path.join('..', '..'))
if _REPO_ROOT not in _sys.path:
    _sys.path.insert(0, _REPO_ROOT)
from metrics_utils import ExtraMetrics


## 2. Load Preprocessed Data


In [2]:
with open('../data/processed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_train = data['X_train']
X_test = data['X_test']
y_train = data['y_train']
y_test = data['y_test']


## 3. Train Standard GaussianNB


In [3]:
gnb = GaussianNB()
_t0 = time.perf_counter()
gnb.fit(X_train, y_train)
train_time = time.perf_counter() - _t0
# Score the non-private fit on the training partition too, so the
# generalisation gap can be read next to the held-out metrics.
y_train_pred = gnb.predict(X_train)
y_pred = gnb.predict(X_test)

extra = ExtraMetrics()
accuracy = accuracy_score(y_test, y_pred)
extra.add(y_test, y_pred, model=gnb, X=X_test, train_true=y_train, train_pred=y_train_pred, train_time=train_time)
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print(f"Accuracy:  {accuracy:.4f}")
print(f"F1-Score:  {f1:.4f}")


Accuracy:  0.5469
F1-Score:  0.1714


## 4. Save Baseline Results


In [4]:
os.makedirs('output', exist_ok=True)

baseline_results = {
    'model': 'Standard Gaussian Naive Bayes',
    'dataset': 'Gallstone',
    'accuracy': accuracy,
    'f1_score': f1,
    'precision': precision,
    'recall': recall
}
baseline_results.update(extra.means())

pd.DataFrame([baseline_results]).to_csv('output/baseline_results.csv', index=False)
with open('output/std_gnb_report.json', 'w') as f:
    json.dump(baseline_results, f, indent=4)

print("Baseline saved.")

# Export model for MIA
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..', '..')))
from model_exporter import export_model
export_model(gnb, 'output/model/std_gnb_model.pkl')


print("Added metrics:")
print(extra.describe())

Baseline saved.


Model successfully exported to output/model/std_gnb_model.pkl
Added metrics:
  Train Acc:      0.5765 ± 0.0000
  Balanced Acc:   0.5469 ± 0.0000
  AUROC:          0.7520 ± 0.0000
  Train Time (s): 0.0006 ± 0.0000
